# EDA: Synthetic Service History Dataset

This notebook explores the synthetic dataset generated by `training/synthesize.py`.
It covers:
- Basic shape and stats
- Distribution of input features (`months_driven`, `total_kms_driven`)
- Distribution of targets (`days_until_service`, `kms_until_service`)
- Driver profile breakdown
- Correlation between features and targets
- Class balance of `trigger` (time vs km)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('../data/processed/service_history.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

In [ ]:
print('Null counts:')
print(df.isnull().sum())

print('\nTrigger distribution (time vs km):')
print(df['trigger'].value_counts())

print('\nDriver profile distribution:')
print(df['driver_profile'].value_counts())

## Feature distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['months_driven'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(6, color='red', linestyle='--', label='Time threshold (6 months)')
axes[0].set_title('months_driven')
axes[0].set_xlabel('Months')
axes[0].legend()

axes[1].hist(df['total_kms_driven'], bins=40, color='coral', edgecolor='white')
axes[1].axvline(10_000, color='red', linestyle='--', label='KM threshold (10,000)')
axes[1].set_title('total_kms_driven')
axes[1].set_xlabel('Kilometres')
axes[1].legend()

plt.suptitle('Input Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## Target distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['days_until_service'], bins=40, color='mediumseagreen', edgecolor='white')
axes[0].set_title('days_until_service (target)')
axes[0].set_xlabel('Days')

axes[1].hist(df['kms_until_service'], bins=40, color='mediumpurple', edgecolor='white')
axes[1].set_title('kms_until_service (target)')
axes[1].set_xlabel('Kilometres')

plt.suptitle('Target Distributions', y=1.02)
plt.tight_layout()
plt.show()

## Profiles vs km driven

In [ ]:
colors = {'light': 'steelblue', 'average': 'coral', 'heavy': 'mediumseagreen'}

plt.figure(figsize=(8, 5))
for profile, group in df.groupby('driver_profile'):
    plt.scatter(
        group['months_driven'],
        group['total_kms_driven'],
        alpha=0.3,
        s=10,
        color=colors[profile],
        label=profile,
    )

plt.axhline(10_000, color='red', linestyle='--', linewidth=1, label='KM threshold')
plt.axvline(6, color='orange', linestyle='--', linewidth=1, label='Time threshold')
plt.xlabel('months_driven')
plt.ylabel('total_kms_driven')
plt.title('Driver Profiles in Feature Space')
plt.legend()
plt.tight_layout()
plt.show()

## Correlation heatmap

In [ ]:
numeric_cols = ['months_driven', 'total_kms_driven', 'days_until_service', 'kms_until_service']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=30, ha='right')
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Key takeaways

Note the strong **negative correlation** between input features and targets — the more months/km already driven, the fewer days/km until service is due. This is exactly what a regression model should learn.

The distribution of `days_until_service` is skewed right with a spike at 0 (overdue vehicles). `kms_until_service` similarly has a spike at 0 for heavy drivers who hit the km threshold first.

The scatter plot shows that the two thresholds (6 months, 10,000 km) form a natural decision boundary — heavy drivers cross the km line first, light drivers cross the time line first. The model's job is to learn which quadrant a driver is in from just two features.